<a href="https://colab.research.google.com/github/mariemghorbel1/DeepLearning/blob/main/projetdeep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import DonutProcessor, VisionEncoderDecoderModel
from PIL import Image

# 1. Charger le modèle et le processor
model_name = "naver-clova-ix/donut-base-finetuned-rvlcdip"
processor = DonutProcessor.from_pretrained(model_name)
model = VisionEncoderDecoderModel.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# 2. Charger et redimensionner l'image
image_path = "/facture.png"  # Remplace par ton chemin
image = Image.open(image_path).convert("RGB")
image = image.resize((1280, 960))  # Peut aider

# 3. Préparer les entrées
task_prompt = "<s_rvlcdip>"
decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids.to(device)
pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

# 4. Générer
outputs = model.generate(
    pixel_values,
    decoder_input_ids=decoder_input_ids,
    max_length=16,
    pad_token_id=processor.tokenizer.pad_token_id,
    eos_token_id=processor.tokenizer.eos_token_id,
    bad_words_ids=[[processor.tokenizer.unk_token_id]],
)

# 5. Décoder
decoded = processor.batch_decode(outputs, skip_special_tokens=True)[0]
predicted_class = decoded.replace("<s_rvlcdip>", "").strip()

# 6. Afficher
if not predicted_class:
    print("❗ Aucune classe prédite. Le document est peut-être hors distribution.")
else:
    print(f"🧾 Document classifié comme : {predicted_class}")


❗ Aucune classe prédite. Le document est peut-être hors distribution.


In [ ]:
# 📦 Importation des bibliothèques nécessaires
import torch  # Pour gérer les tenseurs et utiliser GPU/CPU
import re  # Pour extraire la classe via une expression régulière
from transformers import DonutProcessor, VisionEncoderDecoderModel  # Pour charger le modèle Donut
from PIL import Image  # Pour ouvrir et manipuler l’image du document

# 🧠 1. Charger le modèle pré-entraîné Donut + son processor
model_name = "naver-clova-ix/donut-base-finetuned-rvlcdip"  # Modèle entraîné sur le dataset RVL-CDIP
processor = DonutProcessor.from_pretrained(model_name)  # Prépare les données pour le modèle
model = VisionEncoderDecoderModel.from_pretrained(model_name)  # Charge le modèle complet

# ⚙️ 2. Choisir l'appareil : GPU (cuda) ou CPU
device = "cuda" if torch.cuda.is_available() else "cpu"  # Utilise GPU si dispo
model.to(device)  # Envoie le modèle vers l'appareil choisi

# 🖼️ 3. Charger l'image du document à classer
image_path = "/Capture d'écran 2025-08-05 110414.png"  # 📝 Remplace par le chemin de ton image
image = Image.open(image_path).convert("RGB")  # Ouvre l’image et la convertit en RGB
image = image.resize((1280, 960))  # Redimensionne (taille optimale pour Donut)

# 🏷️ 4. Créer le prompt de tâche pour la classification RVL-CDIP
task_prompt = "<s_rvlcdip>"  # Ce prompt indique la tâche à effectuer : classification
decoder_input_ids = processor.tokenizer(
    task_prompt,
    add_special_tokens=False,
    return_tensors="pt"
).input_ids.to(device)  # Transforme le prompt en ID utilisable par le modèle

# 📸 5. Transformer l’image en tenseur
pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)  # Prépare l’image pour le modèle

# 🔮 6. Générer une prédiction à partir de l’image
outputs = model.generate(
    pixel_values,  # Image encodée
    decoder_input_ids=decoder_input_ids,  # Prompt de tâche
    max_length=32,  # Longueur maximale de la réponse (évite les coupures)
    pad_token_id=processor.tokenizer.pad_token_id,  # Token de remplissage
    eos_token_id=processor.tokenizer.eos_token_id,  # Token de fin
    bad_words_ids=[[processor.tokenizer.unk_token_id]],  # Empêche les tokens inconnus
)

# 📤 7. Décoder la sortie brute (inclut les balises spéciales)
raw_output = processor.batch_decode(outputs, skip_special_tokens=False)[0]
print("🔍 Sortie brute (avec tokens spéciaux) :", raw_output)  # Affiche le texte généré brut

# 🧹 8. Fonction pour extraire la classe depuis la sortie
def extract_class_from_raw(raw_output):
    """
    Extrait la classe entre les balises <s_class> et </s_class>.
    Exemple d’entrée : <s_rvlcdip><s_class><letter/></s_class></s>
    Résultat : 'letter'
    """
    match = re.search(r"<s_class><(.*?)/></s_class>", raw_output)
    if match:
        return match.group(1)  # Retourne le texte entre les balises
    return None  # Retourne rien si pas de correspondance

# 📦 9. Appliquer l’extraction + affichage du résultat
predicted_class = extract_class_from_raw(raw_output)  # Extrait la classe prédite

# 📢 10. Afficher le résultat final
if not predicted_class:
    print("❗ Aucune classe prédite. Le document est peut-être hors distribution.")  # Aucun résultat
else:
    print(f"🧾 Document classifié comme : {predicted_class}")  # Affiche la classe (ex : letter, form, etc.)


🔍 Sortie brute (avec tokens spéciaux) : <s_rvlcdip><s_class><letter/></s_class></s>
🧾 Document classifié comme : letter


In [ ]:
#     DONUT & TESSERACT
import torch
import re
from transformers import DonutProcessor, VisionEncoderDecoderModel
from PIL import Image

# 1. Charger le modèle et le processor
model_name = "naver-clova-ix/donut-base-finetuned-rvlcdip"
processor = DonutProcessor.from_pretrained(model_name)
model = VisionEncoderDecoderModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# 2. Charger l'image
image_path = "/facture.png"  # 🔁 Remplace par ton propre chemin
image = Image.open(image_path).convert("RGB")
image = image.resize((1280, 960))  # 🔧 Taille recommandée pour Donut

# 3. Préparer les entrées
task_prompt = "<s_rvlcdip>"
decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids.to(device)
pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

# 4. Générer la prédiction
outputs = model.generate(
    pixel_values,
    decoder_input_ids=decoder_input_ids,
    max_length=32,  # 🔁 Allongé pour éviter de tronquer
    pad_token_id=processor.tokenizer.pad_token_id,
    eos_token_id=processor.tokenizer.eos_token_id,
    bad_words_ids=[[processor.tokenizer.unk_token_id]],
)

# 5. Afficher la sortie brute
raw_output = processor.batch_decode(outputs, skip_special_tokens=False)[0]
print("🔍 Sortie brute (avec tokens spéciaux) :", raw_output)

# 6. Fonction d’extraction de la classe
def extract_class_from_raw(raw_output):
    """
    Extrait la classe prédite à partir de la sortie brute générée par le modèle Donut.
    Exemple : <s_rvlcdip><s_class><letter/></s_class></s> → 'letter'
    """
    match = re.search(r"<s_class><(.*?)/></s_class>", raw_output)
    if match:
        return match.group(1)
    return None

# 7. Nettoyage final + affichage
predicted_class = extract_class_from_raw(raw_output)

if not predicted_class:
    print("❗ Aucune classe prédite. Le document est peut-être hors distribution.")
else:
    print(f"🧾 Document classifié comme : {predicted_class}")


🔍 Sortie brute (avec tokens spéciaux) : <s_rvlcdip><s_class><invoice/></s_class></s>
🧾 Document classifié comme : invoice


In [ ]:
# =============================
# 📦 1. Installation des dépendances
# =============================
!sudo apt install tesseract-ocr -y
!pip install pytesseract Pillow transformers fuzzywuzzy python-Levenshtein --quiet
!apt-get update
!apt-get install -y tesseract-ocr
!apt-get install -y tesseract-ocr-fra

import os
os.environ['TESSDATA_PREFIX'] = '/usr/share/tesseract-ocr/4.00/tessdata/'


# =============================
# 📁 2. Configuration : Chemin de l’image à analyser
# =============================
image_path = "/Capture d'écran 2025-07-08 100756.png"  # <-- 🔁 Modifie ce chemin avec celui de ton image

# =============================
# 📥 3. Imports
# =============================
from PIL import Image
import pytesseract
import re
import torch
from fuzzywuzzy import fuzz
from transformers import DonutProcessor, VisionEncoderDecoderModel
from datetime import datetime
import json

# =============================
# 📷 4. Chargement de l’image
# =============================
image = Image.open(image_path).convert("RGB")
image.thumbnail((800, 800))
image.show()

# =============================
# 🔠 5. Définition des mots-clés et fonctions de parsing
# =============================
keywords = {
    "facture": ["facture", "Facture", "FACTURE", "fact ur", "fact.ure", "factur"],
    "date": ["date", "Date", "DATE", "dte", "dat", "dt", "dare", "datte"],
    "total": ["total", "Total", "TOTAL", "montant total", "montant", "ttc"],
    "tva": ["tva", "TVA", "tv4", "valeur ajoutée", "taxe"],
    "fournisseur": ["fournisseur", "Fournisseur", "FOURNISSEUR", "fornisseur"],
    "client": ["client", "Client", "CLIENT", "clien", "client final"],
    "paiement": ["paiement", "mode de paiement", "payment", "payement"],
    "iban": ["iban", "IBAN", "rib", "RIB", "compte bancaire"]
}

def is_similar(text, keywords_list, threshold=80):
    return any(fuzz.partial_ratio(text.lower(), kw.lower()) >= threshold for kw in keywords_list)

# =============================
# 🤖 6. Extraction via Donut
# =============================
processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base-finetuned-cord-v2", use_fast=False)
model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base-finetuned-cord-v2")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
decoder_input_ids = processor.tokenizer("<s_cord-v2>", add_special_tokens=False, return_tensors="pt").input_ids.to(device)

outputs = model.generate(
    pixel_values,
    decoder_input_ids=decoder_input_ids,
    max_length=model.decoder.config.max_position_embeddings,
    pad_token_id=processor.tokenizer.pad_token_id,
    eos_token_id=processor.tokenizer.eos_token_id,
    bad_words_ids=[[processor.tokenizer.unk_token_id]],
)

sequence = processor.batch_decode(outputs)[0]
sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
donut_data = processor.token2json(sequence)

def parse_donut_data(data):
    result = {
        "date": "",
        "total": "",
        "tva": "",
        "numero_facture": "",
        "fournisseur": "",
        "client": "",
        "mode_paiement": "",
        "iban_rib": ""
    }
    try:
        menu = data.get("menu", [])
        sub_total = data.get("sub_total", {})
        total = data.get("total", {})
        if menu:
            result["fournisseur"] = menu[0].get("nm", "")
            result["numero_facture"] = menu[1].get("price", "") if len(menu) > 1 else ""
        result["total"] = total.get("total_price", "")
        result["tva"] = sub_total.get("tax_price", "")
        result["mode_paiement"] = total.get("creditcardprice", "")
    except Exception as e:
        print(f"Erreur parsing Donut: {e}")
    return result

donut_result = parse_donut_data(donut_data)

# =============================
# 🔡 7. Extraction via Tesseract
# =============================
raw_text = pytesseract.image_to_string(image, lang="fra")

def extract_fields_tesseract(text):
    result = {
        "date": "",
        "total": "",
        "tva": "",
        "numero_facture": "",
        "fournisseur": "",
        "client": "",
        "mode_paiement": "",
        "iban_rib": ""
    }

    mois_fr = ["janvier", "février", "fevrier", "mars", "avril", "mai", "juin",
               "juillet", "août", "aout", "septembre", "octobre", "novembre", "décembre", "decembre"]
    mois_to_num = {
        "janvier": "01", "février": "02", "fevrier": "02", "mars": "03", "avril": "04", "mai": "05", "juin": "06",
        "juillet": "07", "août": "08", "aout": "08", "septembre": "09", "octobre": "10", "novembre": "11", "décembre": "12", "decembre": "12"
    }

    lines = text.split('\n')
    for line in lines:
        clean_line = re.sub(r'[^\w\s:/.-]', '', line.strip())
        if not clean_line:
            continue
        lower = clean_line.lower()

        if not result["date"]:
            m = re.search(r"\b(\d{2}[/-]\d{2}[/-]\d{4})\b", lower)
            if m:
                result["date"] = m.group(1)
            else:
                for mois in mois_fr:
                    m2 = re.search(rf"\b(\d{{1,2}})\s+{mois}\s+(\d{{4}})\b", lower)
                    if m2:
                        j = m2.group(1).zfill(2)
                        m_num = mois_to_num[mois]
                        a = m2.group(2)
                        result["date"] = f"{j}/{m_num}/{a}"
                        break

        if not result["numero_facture"] and is_similar(lower, keywords["facture"]):
            m = re.search(r"\b\d{4,}\b", lower)
            if m:
                result["numero_facture"] = m.group()

        if not result["total"] and is_similar(lower, keywords["total"]):
            m = re.findall(r"\d+[.,]?\d*\s?(?:€|eur|dt|dhs)?", lower)
            if m:
                montant = re.findall(r"\d+[.,]?\d*", m[-1])
                result["total"] = (montant[-1] if montant else "") + " €"

        if not result["tva"] and is_similar(lower, keywords["tva"]):
            m = re.search(r"\d{1,2}[.,]?\d{0,2}\s?%", lower)
            if m:
                result["tva"] = m.group()

        if not result["fournisseur"] and is_similar(lower, keywords["fournisseur"]):
            result["fournisseur"] = clean_line

        if not result["client"] and is_similar(lower, keywords["client"]):
            result["client"] = clean_line

        if not result["mode_paiement"] and is_similar(lower, keywords["paiement"]):
            for mot in ["virement", "chèque", "cheque", "carte", "espèces", "especes", "prélèvement", "paypal"]:
                if mot in lower:
                    result["mode_paiement"] = mot.capitalize()
                    break

        if not result["iban_rib"] and is_similar(lower, keywords["iban"]):
            m = re.search(r"\b[A-Z]{2}\d{2}\s?(?:\d{4}\s?){2,}\d{1,4}\b", clean_line.replace(" ", ""))
            if m:
                result["iban_rib"] = m.group()

    return result

tesseract_result = extract_fields_tesseract(raw_text)

# =============================
# 🔄 8. Fusion et affichage final
# =============================
from pprint import pprint

print("===== Résultats combinés (Donut + Tesseract) =====\n")
combined_result = {}
for key in tesseract_result:
    combined_result[key] = donut_result.get(key) or tesseract_result.get(key) or "Non trouvé"

pprint(combined_result, width=100)


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured f

In [ ]:
# =============================
# 📦 0. Installation des dépendances dans Colab
# =============================

!apt-get update  # Met à jour la liste des paquets Linux
!apt-get install -y tesseract-ocr tesseract-ocr-fra  # Installe Tesseract avec le support français
!pip install pytesseract  # Lib Python pour utiliser Tesseract
!pip install fuzzywuzzy python-Levenshtein  # fuzzywuzzy = comparaison de similarité de texte

# =============================
# 📚 1. Importation des bibliothèques
# =============================
import pytesseract  # OCR basé sur Tesseract
from fuzzywuzzy import fuzz  # Mesurer la similarité entre 2 chaînes
import torch  # Librairie pour manipuler les modèles IA (GPU/CPU)
import re  # Expressions régulières pour rechercher du texte
from transformers import DonutProcessor, VisionEncoderDecoderModel  # Donut et outils HuggingFace
from PIL import Image  # Manipuler les images
from pprint import pprint  # Affichage clair des dictionnaires

# =============================
# 🧠 2. Chargement du modèle Donut de classification (RVL-CDIP)
# =============================
model_class_name = "naver-clova-ix/donut-base-finetuned-rvlcdip"  # Nom du modèle Donut entraîné sur RVL-CDIP
processor_class = DonutProcessor.from_pretrained(model_class_name)  # Préprocesseur pour préparer images + texte
model_class = VisionEncoderDecoderModel.from_pretrained(model_class_name)  # Modèle de classification

# Utilisation GPU si dispo, sinon CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model_class.to(device)

# =============================
# 📷 3. Charger l'image à traiter
# =============================
image_path = "/content/Capture d'écran 2025-07-08 100732.png"  # 🔁 Chemin de ton image
image = Image.open(image_path).convert("RGB")  # Ouvre l'image en mode couleur
image_resized = image.resize((1280, 960))  # Redimensionner (format recommandé pour Donut)

# =============================
# 🏷️ 4. Préparer le prompt pour classification
# =============================
task_prompt = "<s_rvlcdip>"  # Indique à Donut qu'on fait une classification RVL-CDIP
decoder_input_ids = processor_class.tokenizer(
    task_prompt,  # Notre prompt
    add_special_tokens=False,  # Pas besoin d’ajouter les tokens spéciaux du modèle
    return_tensors="pt"  # Retourner au format tensor PyTorch
).input_ids.to(device)  # Envoyer sur GPU/CPU

# =============================
# 🔮 5. Prédire la classe du document
# =============================
pixel_values = processor_class(image_resized, return_tensors="pt").pixel_values.to(device)  # Convertir image en tensor
outputs = model_class.generate(
    pixel_values,  # Image traitée
    decoder_input_ids=decoder_input_ids,  # Prompt en entrée
    max_length=32,  # Longueur max de sortie
    pad_token_id=processor_class.tokenizer.pad_token_id,  # Token de remplissage
    eos_token_id=processor_class.tokenizer.eos_token_id,  # Fin de séquence
    bad_words_ids=[[processor_class.tokenizer.unk_token_id]],  # Interdit les tokens inconnus
)

raw_output = processor_class.batch_decode(outputs, skip_special_tokens=False)[0]  # Décodage du texte brut

# Fonction pour extraire uniquement la classe prédite
def extract_class_from_raw(raw_output):
    """Extrait la classe prédite depuis la sortie brute du modèle."""
    match = re.search(r"<s_class><(.*?)/></s_class>", raw_output)
    return match.group(1) if match else None

predicted_class = extract_class_from_raw(raw_output)
print(f"🧾 Classe prédite : {predicted_class}")

# =============================
# 📌 6. Si ce n'est pas une facture → arrêter
# =============================
if predicted_class not in ["invoice", "facture"]:
    print("❌ Document non reconnu comme facture. Aucune extraction.")
    exit()  # Stoppe l'exécution

print("✅ Facture détectée → extraction des champs...")

# =============================
# 🤖 7. Extraction avec Donut (CORD-v2)
# =============================
processor_extract = DonutProcessor.from_pretrained("naver-clova-ix/donut-base-finetuned-cord-v2", use_fast=False)
model_extract = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base-finetuned-cord-v2")
model_extract.to(device)

# Préparation image pour extraction
pixel_values = processor_extract(image, return_tensors="pt").pixel_values.to(device)
decoder_input_ids = processor_extract.tokenizer("<s_cord-v2>", add_special_tokens=False, return_tensors="pt").input_ids.to(device)

# Génération de la sortie Donut
outputs = model_extract.generate(
    pixel_values,
    decoder_input_ids=decoder_input_ids,
    max_length=model_extract.decoder.config.max_position_embeddings,
    pad_token_id=processor_extract.tokenizer.pad_token_id,
    eos_token_id=processor_extract.tokenizer.eos_token_id,
    bad_words_ids=[[processor_extract.tokenizer.unk_token_id]],
)

sequence = processor_extract.batch_decode(outputs)[0]  # Décoder la sortie en texte
sequence = sequence.replace(processor_extract.tokenizer.eos_token, "").replace(processor_extract.tokenizer.pad_token, "")
donut_data = processor_extract.token2json(sequence)  # Convertir en JSON

# Fonction pour parser les champs
def parse_donut_data(data):
    """Parse les champs extraits par Donut."""
    result = {
        "date": "",
        "total": "",
        "tva": "",
        "numero_facture": "",
        "fournisseur": "",
        "client": "",
        "mode_paiement": "",
        "iban_rib": ""
    }
    try:
        menu = data.get("menu", [])
        sub_total = data.get("sub_total", {})
        total = data.get("total", {})
        if menu:
            result["fournisseur"] = menu[0].get("nm", "")
            result["numero_facture"] = menu[1].get("price", "") if len(menu) > 1 else ""
        result["total"] = total.get("total_price", "")
        result["tva"] = sub_total.get("tax_price", "")
        result["mode_paiement"] = total.get("creditcardprice", "")
    except Exception as e:
        print(f"Erreur parsing Donut: {e}")
    return result

donut_result = parse_donut_data(donut_data)

# =============================
# 🔡 8. Extraction avec Tesseract OCR
# =============================
# Liste de mots-clés pour repérer les infos
  keywords = {
      "facture": ["facture", "Facture", "FACTURE"],
      "date": ["date", "Date", "DATE"],
      "total": ["total", "Total", "TOTAL", "montant total", "montant", "ttc"],
      "tva": ["tva", "TVA", "valeur ajoutée", "taxe"],
      "fournisseur": ["fournisseur", "Fournisseur", "FOURNISSEUR"],
      "client": ["client", "Client", "CLIENT"],
      "paiement": ["paiement", "mode de paiement"],
      "iban": ["iban", "IBAN", "rib", "RIB"]
}

# Fonction pour vérifier similarité texte / mot-clé
def is_similar(text, keywords_list, threshold=80):
    return any(fuzz.partial_ratio(text.lower(), kw.lower()) >= threshold for kw in keywords_list)

# OCR sur l'image
raw_text = pytesseract.image_to_string(image, lang="fra")

# Extraction des infos à partir du texte OCR
def extract_fields_tesseract(text):
    """Extraction des champs avec Tesseract."""
    result = {k: "" for k in ["date", "total", "tva", "numero_facture", "fournisseur", "client", "mode_paiement", "iban_rib"]}
    lines = text.split('\n')
    for line in lines:
        clean_line = re.sub(r'[^\w\s:/.-]', '', line.strip())
        if not clean_line:
            continue
        lower = clean_line.lower()
        if not result["total"] and is_similar(lower, keywords["total"]):
            m = re.findall(r"\d+[.,]?\d*", lower)
            if m:
                result["total"] = m[-1] + " €"
    return result

tesseract_result = extract_fields_tesseract(raw_text)

# =============================
# 🔄 9. Fusion des résultats Donut + Tesseract
# =============================
combined_result = {}
for key in tesseract_result:
    combined_result[key] = donut_result.get(key) or tesseract_result.get(key) or "Non trouvé"

print("\n===== Résultats combinés =====")
pprint(combined_result, width=100)


Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/536 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/806M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/806M [00:00<?, ?B/s]


===== Résultats combinés =====
{'client': 'Non trouvé',
 'date': 'Non trouvé',
 'fournisseur': 'Lorem ipsum dclour sit amet',
 'iban_rib': 'Non trouvé',
 'mode_paiement': 'Non trouvé',
 'numero_facture': '500.00',
 'total': '0,00 €',
 'tva': 'Non trouvé'}
